# Redes LSTM con Keras para procesamiento secuencial de texto

**Materiales desarrollados por Matías Barreto, 2025**

**Tecnicatura en Ciencia de Datos - IFTS**

**Asignatura:** Procesamiento de Lenguaje Natural

---

## Objetivo

Pasar de modelos que ignoran el orden de las palabras a un modelo secuencial que procese texto token por token y mantenga memoria del contexto.

## Resultados de aprendizaje

Al final de este notebook vas a poder:

1. Explicar por qué BoW y MLP no alcanzan para capturar orden temporal.
2. Convertir frases en secuencias de enteros con `Tokenizer`.
3. Aplicar `padding` para uniformar longitudes.
4. Construir una red LSTM paso a paso en Keras.
5. Interpretar una validación durante el entrenamiento y una pequeña prueba externa.

## Relación con el notebook anterior

En `04` usamos una red multicapa, pero la entrada seguía siendo un vector sin orden. Ahora el dato cambia de forma: ya no entra como bolsa de palabras, sino como secuencia.

## Introducción

Una LSTM sigue procesando números, pero ahora esos números respetan el orden en que aparecen las palabras. Eso permite capturar fenómenos que en un modelo feedforward suelen perderse, como negaciones, intensificadores y dependencias locales.


---

## 1. Instalación e Importación de Librerías

En Google Colab, TensorFlow/Keras ya viene instalado.

In [ ]:
# Si necesitás instalar TensorFlow (descomentá)
# !pip install tensorflow

# NumPy para operaciones numéricas
import numpy as np

# TensorFlow y Keras
import tensorflow as tf
from tensorflow import keras

# Keras Sequential: API para apilar capas
from tensorflow.keras.models import Sequential

# Capas de Keras
# Embedding: Capa que aprende representaciones vectoriales de palabras
# LSTM: Capa recurrente con memoria a largo plazo
# Dense: Capa fully connected (como en MLP)
from tensorflow.keras.layers import Embedding, LSTM, Dense

# Preprocesamiento de texto
# Tokenizer: Convierte texto en secuencias de enteros
# pad_sequences: Uniformiza longitud de secuencias
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Matplotlib para visualizaciones
import matplotlib.pyplot as plt

# Fijamos semillas para reproducibilidad
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow versión: {tf.__version__}")
print(f"Keras versión: {keras.__version__}")
print(f"GPU disponible: {len(tf.config.list_physical_devices('GPU')) > 0}")

---

## 2. Dataset: Análisis de Sentimiento en Español Rioplatense

Usamos el mismo corpus que en el notebook de PyTorch para comparar resultados.

In [ ]:
# Corpus de frases en español rioplatense
# Etiqueta: 1 = Positivo, 0 = Negativo
frases = [
    # Positivas
    "La verdad, este lugar está bárbaro. Muy recomendable.",
    "Qué buena onda la atención, volvería sin dudarlo.",
    "Me encantó la comida, aunque la música estaba muy fuerte.",
    "Todo excelente. Atención de diez.",
    "Muy conforme con el resultado final.",
    "Superó mis expectativas, gracias.",
    "El mejor asado que probé en mucho tiempo.",
    "Excelente relación precio-calidad, muy recomendable.",
    "La atención fue impecable, muy atentos.",
    "Me gustó mucho el ambiente tranquilo.",

    # Negativas
    "Una porquería de servicio, nunca más vuelvo.",
    "El envío fue lento y el producto llegó dañado. Qué desastre.",
    "Qué estafa, me arrepiento de haber comprado.",
    "No me gustó para nada la experiencia.",
    "No lo recomiendo, mala calidad.",
    "Malísima atención, el mozo tenía mala onda.",
    "Tardaron dos horas en entregar, llegó todo frío.",
    "Me cobraron de más y encima se hicieron los giles.",
    "La carne estaba pasada, casi no se podía comer.",
    "Pésima experiencia, no vuelvo más."
]

# Etiquetas: 1=positivo, 0=negativo
etiquetas = np.array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1,  # 10 positivas
                      0, 0, 0, 0, 0, 0, 0, 0, 0, 0])  # 10 negativas

print(f"Corpus: {len(frases)} frases")
print(f"Distribución: {np.bincount(etiquetas)}")
print(f"\nEjemplos:")
print(f"[+] {frases[0]}")
print(f"[-] {frases[10]}")

---

## 3. Tokenización: de texto a secuencias de enteros

El `Tokenizer` de Keras cumple dos tareas:

1. Construye un vocabulario (palabra -> índice).
2. Convierte cada frase en una secuencia de enteros.

Antes de automatizar, hacé esta prueba mental con una frase corta:

- Elegí dos o tres palabras del corpus.
- Asignales números.
- Reescribí una frase reemplazando cada palabra por su índice.

Eso es exactamente lo que vamos a hacer ahora, pero a escala del corpus completo.


In [ ]:
# Si necesitás instalar TensorFlow (descomentá)
# !pip install tensorflow

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import matplotlib.pyplot as plt

np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow versión: {tf.__version__}")
print(f"Keras versión: {keras.__version__}")
print(f"GPU disponible: {len(tf.config.list_physical_devices('GPU')) > 0}")

# Configuración del tokenizer
# `oov_token` representa palabras que no estuvieron en el entrenamiento.
tokenizer = Tokenizer(oov_token="<OOV>")

# Construimos el vocabulario a partir del corpus.
tokenizer.fit_on_texts(frases)

vocab_size = len(tokenizer.word_index) + 1
print(f"
Tamaño del vocabulario: {vocab_size} palabras")
print("Primeras 15 palabras del vocabulario:")

primeras_palabras = []
contador_palabras = 0
for palabra, indice in tokenizer.word_index.items():
    primeras_palabras.append((palabra, indice))
    contador_palabras += 1
    if contador_palabras == 15:
        break

print(primeras_palabras)

secuencias = tokenizer.texts_to_sequences(frases)
print("
Ejemplo de secuencia:")
print(f"Texto original: '{frases[0]}'")
print(f"Secuencia: {secuencias[0]}")
print(f"Longitud: {len(secuencias[0])} tokens")

# Reconstruimos la primera secuencia para verificar el mapeo.
indice_a_palabra = {}
for palabra, indice in tokenizer.word_index.items():
    indice_a_palabra[indice] = palabra

palabras_reconstruidas = []
for indice in secuencias[0]:
    palabra = indice_a_palabra.get(indice, '<OOV>')
    palabras_reconstruidas.append(palabra)

texto_reconstruido = ' '.join(palabras_reconstruidas)
print(f"Reconstrucción aproximada: '{texto_reconstruido}'")


---

## 4. Padding: uniformar longitudes sin perder la secuencia

Las frases no tienen todas la misma cantidad de palabras. Para poder entrenar en lotes, necesitamos llevarlas a una longitud común.

En este notebook vamos a usar `post-padding`, es decir, agregaremos ceros al final. Primero vamos a medir cuánto ocupa cada secuencia y después elegiremos un `maxlen` razonable.


In [ ]:
longitudes = []
for secuencia in secuencias:
    longitud = len(secuencia)
    longitudes.append(longitud)

print("Estadísticas de longitudes:")
print(f"Mínima: {min(longitudes)} tokens")
print(f"Máxima: {max(longitudes)} tokens")
print(f"Promedio: {np.mean(longitudes):.1f} tokens")

maxlen = max(longitudes)
print(f"
Usaremos maxlen = {maxlen}")

X = pad_sequences(secuencias, maxlen=maxlen, padding='post', truncating='post')
y = etiquetas.astype(np.float32)

print(f"
Forma de X: {X.shape} (muestras x tokens)")
print("Ejemplo de secuencia con padding:")
print(f"Original ({len(secuencias[0])} tokens): {secuencias[0]}")
print(f"Con padding ({maxlen} tokens): {X[0]}")
print("
Lectura: los ceros agregados al final no son palabras; solo completan la longitud.")


---

## 5. Embeddings: Representaciones Vectoriales de Palabras

### ¿Qué son los embeddings?

Son **representaciones vectoriales densas** de palabras en un espacio continuo de baja dimensión (típicamente 50-300 dimensiones).

### Ventajas sobre one-hot encoding:

**One-hot** (vocabulario de 10,000 palabras):
```
"gato" → [0, 0, 1, 0, 0, ..., 0]  # Vector de 10,000 elementos
"perro" → [0, 1, 0, 0, 0, ..., 0]
```
- Muy disperso (sparse)
- No captura similitud semántica
- Ineficiente computacionalmente

**Embeddings** (dimensión 16):
```
"gato"  → [0.2, -0.5, 0.8, 0.1, ..., -0.3]  # Vector de 16 elementos
"perro" → [0.3, -0.4, 0.7, 0.2, ..., -0.2]  # Similar a "gato"
"casa"  → [-0.8, 0.6, -0.1, 0.9, ..., 0.5]  # Diferente
```
- Denso (compact)
- Captura relaciones semánticas
- Palabras similares tienen vectores similares

### Embeddings aprendidos vs. pre-entrenados:

**Aprendidos (lo que haremos aquí):**
- Se inicializan aleatoriamente
- Se ajustan durante el entrenamiento
- Específicos para nuestra tarea
- Requieren datos suficientes

**Pre-entrenados (Word2Vec, GloVe, FastText):**
- Entrenados en corpus masivos (Wikipedia, Common Crawl)
- Capturan conocimiento lingüístico general
- Útiles con pocos datos
- Los vieron en módulos anteriores del curso

---

## 6. Arquitectura LSTM: Conceptos Fundamentales

### RNN Simple (Recurrent Neural Network)

```
t=1:  x₁ → [RNN] → h₁
              ↓
t=2:  x₂ → [RNN] → h₂
              ↓
t=3:  x₃ → [RNN] → h₃ → output
```

Cada paso recibe:
- Input actual: $x_t$
- Hidden state anterior: $h_{t-1}$

Y produce:
- Nuevo hidden state: $h_t = tanh(W_x x_t + W_h h_{t-1} + b)$

### Problema de las RNN simples: Vanishing Gradient

En secuencias largas, el gradiente se propaga multiplicativamente:
$$\frac{\partial L}{\partial h_1} = \frac{\partial L}{\partial h_T} \cdot \frac{\partial h_T}{\partial h_{T-1}} \cdot ... \cdot \frac{\partial h_2}{\partial h_1}$$

Si cada derivada parcial < 1, el producto tiende a 0 (vanishing gradient).

**Consecuencia:** La RNN "olvida" información temprana, no puede capturar dependencias largas.

### LSTM (Long Short-Term Memory)

LSTM resuelve esto con **celdas de memoria** y **compuertas** (gates):

**Componentes de una celda LSTM:**
1. **Forget gate** (f): ¿Qué información olvidar de la memoria?
2. **Input gate** (i): ¿Qué información nueva agregar?
3. **Cell state** (C): Memoria a largo plazo
4. **Output gate** (o): ¿Qué información exponer?

**Fórmulas (simplificadas):**
```
f_t = σ(W_f · [h_{t-1}, x_t] + b_f)    # Forget gate
i_t = σ(W_i · [h_{t-1}, x_t] + b_i)    # Input gate
C̃_t = tanh(W_C · [h_{t-1}, x_t] + b_C) # Candidate values
C_t = f_t * C_{t-1} + i_t * C̃_t        # Update cell state
o_t = σ(W_o · [h_{t-1}, x_t] + b_o)    # Output gate
h_t = o_t * tanh(C_t)                   # Hidden state
```

**Intuición:**
- La celda de memoria $C_t$ fluye directamente sin multiplicaciones repetidas
- Esto previene vanishing gradient
- Las compuertas aprenden qué recordar y qué olvidar

---

## 7. Construcción del modelo LSTM con Keras

Vamos a definir el modelo capa por capa, sin comprimir toda la arquitectura en una sola lista. La idea es que puedas ver qué entra y qué sale de cada bloque.


In [ ]:
embedding_dim = 16
lstm_units = 32

modelo = Sequential(name='modelo_lstm_sentimiento')

# Capa 1: embeddings aprendidos.
modelo.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        input_length=maxlen,
        name='embedding'
    )
)

# Capa 2: memoria secuencial.
modelo.add(
    LSTM(
        units=lstm_units,
        name='lstm'
    )
)

# Capa 3: salida binaria.
modelo.add(
    Dense(
        units=1,
        activation='sigmoid',
        name='output'
    )
)

print("Modelo LSTM creado:")
print("=" * 70)
modelo.summary()

print("
" + "=" * 70)
print("Lectura de la arquitectura:")
print("-" * 70)
print(f"1. Embedding: transforma índices en vectores de {embedding_dim} dimensiones.")
print(f"2. LSTM: resume la secuencia en un estado oculto de {lstm_units} unidades.")
print("3. Dense: convierte ese resumen en una probabilidad de sentimiento positivo.")


---

## 8. Compilación del Modelo

Especificamos:
- **Loss function**: Binary crossentropy (clasificación binaria)
- **Optimizer**: Adam (adaptativo, eficiente)
- **Metrics**: Accuracy (porcentaje de aciertos)

In [ ]:
# Compilamos el modelo
modelo.compile(
    # Loss: Binary Cross Entropy para clasificación binaria
    loss='binary_crossentropy',

    # Optimizer: Adam con learning rate por defecto (0.001)
    optimizer='adam',

    # Métricas a monitorear durante entrenamiento
    metrics=['accuracy']
)

print("Modelo compilado y listo para entrenar.")
print("\nConfiguración:")
print(f"  Loss: binary_crossentropy")
print(f"  Optimizer: Adam (lr=0.001 por defecto)")
print(f"  Metrics: accuracy")

---

## 9. Entrenamiento del modelo

En este notebook vamos a usar `validation_split=0.25` dentro del corpus de entrenamiento. No reemplaza un benchmark serio, pero sí nos da una primera señal de si el modelo está aprendiendo de forma más saludable que mirando solo el ajuste sobre entrenamiento.


In [ ]:
cantidad_epocas = 40
tamano_lote = 2

print("Iniciando entrenamiento...")
print("=" * 70)

historia = modelo.fit(
    X,
    y,
    epochs=cantidad_epocas,
    batch_size=tamano_lote,
    validation_split=0.25,
    verbose=1
)

print("
" + "=" * 70)
print("Entrenamiento completado.")
print(f"Loss final de entrenamiento: {historia.history['loss'][-1]:.4f}")
print(f"Accuracy final de entrenamiento: {historia.history['accuracy'][-1]:.4f}")
print(f"Loss final de validación: {historia.history['val_loss'][-1]:.4f}")
print(f"Accuracy final de validación: {historia.history['val_accuracy'][-1]:.4f}")


---

## 10. Visualización del Aprendizaje

Graficamos la evolución de loss y accuracy durante el entrenamiento.

In [ ]:
# Extraemos historial de métricas
loss_history = historia.history['loss']
acc_history = historia.history['accuracy']

# Creamos figura con dos subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Subplot 1: Loss
ax1.plot(loss_history, linewidth=2, color='blue')
ax1.set_xlabel('Época', fontsize=12)
ax1.set_ylabel('Loss (Binary Crossentropy)', fontsize=12)
ax1.set_title('Evolución de la Pérdida', fontsize=14)
ax1.grid(True, alpha=0.3)

# Subplot 2: Accuracy
ax2.plot(acc_history, linewidth=2, color='green')
ax2.set_xlabel('Época', fontsize=12)
ax2.set_ylabel('Accuracy', fontsize=12)
ax2.set_title('Evolución de la Precisión', fontsize=14)
ax2.grid(True, alpha=0.3)
ax2.set_ylim([0, 1.05])  # Fijamos rango [0, 1]

plt.tight_layout()
plt.show()

print("Interpretación de las curvas:")
print("=" * 70)
print("- Loss descendente: El modelo está aprendiendo")
print("- Accuracy ascendente: Mejora en predicciones correctas")
print("- Convergencia: Estabilización en valores finales")
print("\nNota: Con más datos, usaríamos un conjunto de validación")
print("para detectar overfitting (train accuracy >> val accuracy).")

---

## 11. Evaluación con prueba externa pequeña

Además del historial de entrenamiento y validación, conviene mirar frases nuevas con etiqueta esperada. Como el corpus es muy chico, esta prueba externa sigue siendo modesta, pero ya evita la mala práctica de evaluar solo sobre entrenamiento.


In [ ]:
frases_prueba_etiquetadas = [
    "No me gustó la atención, bastante mala y lenta.",
    "Muy buena experiencia, todo excelente y rápido.",
    "Una estafa total, no lo recomiendo para nada.",
    "Súper conforme con el servicio, muy atentos.",
    "El lugar está bien, pero la comida fue mala.",
    "La mejor atención que tuve en mucho tiempo."
]

etiquetas_prueba = np.array([0, 1, 0, 1, 0, 1])

secuencias_prueba = tokenizer.texts_to_sequences(frases_prueba_etiquetadas)
X_prueba_etiquetada = pad_sequences(secuencias_prueba, maxlen=maxlen, padding='post')

predicciones_prob = modelo.predict(X_prueba_etiquetada, verbose=0)
predicciones_clase = (predicciones_prob >= 0.5).astype(int).flatten()

aciertos = np.sum(predicciones_clase == etiquetas_prueba)
total = len(etiquetas_prueba)
accuracy = aciertos / total

print("Evaluación sobre prueba externa pequeña:")
print("=" * 70)
for i in range(len(frases_prueba_etiquetadas)):
    prob = predicciones_prob[i][0]
    pred_clase = predicciones_clase[i]
    real_clase = int(etiquetas_prueba[i])

    marca = "OK" if pred_clase == real_clase else "REVISAR"
    sent_real = "Positivo" if real_clase == 1 else "Negativo"
    sent_pred = "Positivo" if pred_clase == 1 else "Negativo"

    print(f"
{marca} '{frases_prueba_etiquetadas[i]}'")
    print(f"  Real esperado: {sent_real} | Predicción: {sent_pred} (prob={prob:.3f})")

print("
" + "=" * 70)
print(f"Accuracy en prueba externa: {aciertos}/{total} = {accuracy:.2%}")
print("
Lectura pedagógica: mirá si el modelo conserva el sentido cuando aparece negación o mezcla de señales.")


---

## 12. Predicción sobre Frases Nuevas

Probamos el modelo con frases que nunca vio durante el entrenamiento.

In [ ]:
# Frases de prueba
frases_prueba = [
    "No me gustó la atención, bastante mala y lenta.",
    "Muy buena experiencia, todo excelente y rápido.",
    "Una estafa total, no lo recomiendo para nada.",
    "Súper conforme con el servicio, muy atentos.",
    "Nada que ver con lo prometido, una decepción.",
    "La mejor atención que tuve en mucho tiempo.",
    "El lugar está bien pero la comida es mala.",
    "Aunque tardaron mucho, la comida estaba excelente."
]

# Preprocesamiento: texto → secuencias → padding
# Usamos el MISMO tokenizer y maxlen que en entrenamiento
secuencias_prueba = tokenizer.texts_to_sequences(frases_prueba)
X_prueba = pad_sequences(secuencias_prueba, maxlen=maxlen, padding='post')

# Predicciones
predicciones_prueba = modelo.predict(X_prueba, verbose=0)

# Mostramos resultados
print("Predicciones sobre frases nuevas:")
print("=" * 70)

for i, frase in enumerate(frases_prueba):
    prob = predicciones_prueba[i][0]
    clase = "POSITIVO" if prob >= 0.5 else "NEGATIVO"
    confianza = prob if prob >= 0.5 else (1 - prob)

    print(f"\nFrase: '{frase}'")
    print(f"Predicción: {clase}")
    print(f"Probabilidad positivo: {prob:.3f}")
    print(f"Confianza: {confianza:.1%}")

---

## 13. Inspección de embeddings aprendidos

La capa `Embedding` produce un vector por palabra. Con este corpus chico no esperamos relaciones semánticas profundas, pero sí podemos mirar si algunas palabras cercanas en función del problema quedan relativamente próximas.


In [ ]:
# Extraemos los pesos de la capa Embedding
embedding_layer = modelo.get_layer('embedding')
embedding_weights = embedding_layer.get_weights()[0]  # Shape: (vocab_size, embedding_dim)

print(f"Matriz de embeddings: {embedding_weights.shape}")
print(f"Cada palabra tiene un vector de {embedding_dim} dimensiones.")

# Función para obtener embedding de una palabra
def get_embedding(palabra):
    idx = tokenizer.word_index.get(palabra.lower())
    if idx is None:
        return None
    return embedding_weights[idx]

# Función para calcular similitud coseno
def similitud_coseno(vec1, vec2):
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

# Analizamos similitudes
print("\n" + "=" * 70)
print("Similitudes semánticas aprendidas:")
print("-" * 70)

pares = [
    ("excelente", "buena"),
    ("excelente", "mala"),
    ("atención", "servicio"),
    ("recomendable", "malísima"),
]

for palabra1, palabra2 in pares:
    emb1 = get_embedding(palabra1)
    emb2 = get_embedding(palabra2)

    if emb1 is not None and emb2 is not None:
        sim = similitud_coseno(emb1, emb2)
        print(f"Similitud('{palabra1}', '{palabra2}'): {sim:.3f}")
    else:
        palabras_faltantes = []
        if emb1 is None:
            palabras_faltantes.append(palabra1)
        if emb2 is None:
            palabras_faltantes.append(palabra2)
        print(f"Palabra(s) no en vocabulario: {', '.join(palabras_faltantes)}")

print("\nInterpretación:")
print("- Similitud cercana a 1: Palabras semánticamente similares")
print("- Similitud cercana a 0: Palabras no relacionadas")
print("- Similitud negativa: Palabras opuestas (ej: bueno/malo)")
print("\nNota: Con más datos, los embeddings capturarían relaciones más ricas.")

---

## 14. Guardar y cargar el modelo (bloque opcional)

Este bloque muestra una práctica útil de trabajo. No es indispensable para entender LSTM, así que podés leerlo como una extensión aplicada.


In [ ]:
modelo.save('lstm_sentiment.h5')
print("Modelo guardado en: lstm_sentiment.h5")

from tensorflow.keras.models import load_model
modelo_cargado = load_model('lstm_sentiment.h5')
print("Modelo cargado correctamente.")

pred_original = modelo.predict(X_prueba_etiquetada, verbose=0)
pred_cargado = modelo_cargado.predict(X_prueba_etiquetada, verbose=0)
igual = np.allclose(pred_original, pred_cargado)
print(f"
¿Predicciones idénticas? {igual}")
print("En un proyecto real, además del modelo conviene persistir el tokenizer y los parámetros de preprocesamiento.")


---

## Guía Teórico-Conceptual

### 1. Comparación: RNN Simple vs. LSTM vs. GRU

| Característica | RNN Simple | LSTM | GRU |
|----------------|------------|------|-----|
| Parámetros | Pocos | Muchos | Moderados |
| Memoria a largo plazo |  Vanishing gradient | OK Cell state | OK Reset/update gates |
| Complejidad | Baja | Alta | Media |
| Velocidad | Rápida | Lenta | Moderada |
| Uso moderno | Obsoleto | Estándar | Alternativa popular |

**GRU (Gated Recurrent Unit):**
- Versión simplificada de LSTM (menos parámetros)
- Combina forget e input gates en un solo "update gate"
- Más rápido de entrenar, rendimiento similar
- Popular en NLP cuando velocidad importa

### 2. Bidirectional LSTM

LSTM procesa secuencia en una dirección:
```
"No me gusta" → [No] → [me] → [gusta]
```

**Bidirectional LSTM** procesa en ambas direcciones:
```
Forward:  [No] → [me] → [gusta]
Backward: [gusta] → [me] → [No]
```

Luego concatena ambos hidden states.

**Ventaja:** Captura contexto antes y después de cada palabra

**Implementación en Keras:**
```python
from tensorflow.keras.layers import Bidirectional

modelo = Sequential([
    Embedding(...),
    Bidirectional(LSTM(32)),  # ← Bidirectional wrapper
    Dense(1, activation='sigmoid')
])
```

**Trade-off:** 2x parámetros, 2x tiempo de entrenamiento

### 3. Return Sequences vs. Return State

**return_sequences=False** (default):
```
Input:  (batch, timesteps, features)
Output: (batch, units)  # Solo hidden state final
```
Útil para clasificación (como en este notebook).

**return_sequences=True**:
```
Input:  (batch, timesteps, features)
Output: (batch, timesteps, units)  # Hidden state en cada timestep
```
Útil para:
- Apilar múltiples LSTMs
- Sequence-to-sequence (traducción)
- Token classification (NER)

**Ejemplo con múltiples LSTMs:**
```python
modelo = Sequential([
    Embedding(...),
    LSTM(64, return_sequences=True),  # Primera LSTM
    LSTM(32),                         # Segunda LSTM
    Dense(1, activation='sigmoid')
])
```

### 4. Embeddings: Aprendidos vs. Pre-entrenados

**¿Cuándo usar embeddings aprendidos?**
- Vocabulario específico del dominio
- Suficientes datos (>10k muestras)
- Tarea muy específica

**¿Cuándo usar pre-entrenados (Word2Vec, GloVe, FastText)?**
- Pocos datos (<1k muestras)
- Vocabulario general
- Transfer learning

**Implementación con pre-entrenados:**
```python
# 1. Cargar embeddings (ej: GloVe)
embeddings_index = {}
with open('glove.6B.100d.txt') as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = coefs

# 2. Crear matriz de embeddings para nuestro vocabulario
embedding_matrix = np.zeros((vocab_size, embedding_dim))
for word, idx in tokenizer.word_index.items():
    embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[idx] = embedding_vector

# 3. Cargar en la capa Embedding
embedding_layer = Embedding(
    vocab_size,
    embedding_dim,
    weights=[embedding_matrix],
    trainable=False  # Congelar embeddings
)
```

### 5. Limitaciones de LSTM

**1. Procesamiento secuencial:**
- No se puede paralelizar (cada paso necesita el anterior)
- Lento en secuencias largas

**2. Dependencias realmente largas:**
- Aunque mejor que RNN simple, LSTM sigue olvidando en textos muy largos (>1000 tokens)

**3. Desbalance de información:**
- Palabras al final influyen más que al inicio
- Bidirectional LSTM mitiga esto

**4. Interpretabilidad:**
- Hidden state es una "caja negra"
- Difícil saber qué información retiene

**Solución moderna: Transformers**
- Attention mechanism: Todas las palabras se ven entre sí
- Paralelizable: Mucho más rápido
- Mejor captura de dependencias largas
- Esto lo veremos en el próximo notebook (BERT, GPT)

### 6. Técnicas para Prevenir Overfitting

**Dropout:**
```python
LSTM(32, dropout=0.2, recurrent_dropout=0.2)
```
- `dropout`: Apaga aleatoriamente inputs
- `recurrent_dropout`: Apaga conexiones recurrentes

**Regularización L2:**
```python
from tensorflow.keras.regularizers import l2

LSTM(32, kernel_regularizer=l2(0.01))
```

**Early Stopping:**
```python
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss', patience=5)
modelo.fit(..., validation_data=(X_val, y_val), callbacks=[early_stop])
```

**Data Augmentation para texto:**
- Sinónimos (usando WordNet)
- Back-translation (traducir y volver a traducir)
- Random insertion/deletion/swap de palabras

### 7. Sequence-to-Sequence (Seq2Seq)

Para tareas como traducción, necesitamos **encoder-decoder**:

```
Encoder LSTM: "Hello" → hidden state h
Decoder LSTM: h → "Hola"
```

**Implementación básica:**
```python
# Encoder
encoder = LSTM(128, return_state=True)
encoder_outputs, state_h, state_c = encoder(encoder_inputs)
encoder_states = [state_h, state_c]

# Decoder
decoder = LSTM(128, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder(decoder_inputs, initial_state=encoder_states)
```

**Attention mechanism:** Mejora crucial (base de Transformers)

---

## Preguntas y Respuestas para Estudio

### Preguntas Conceptuales

**1. ¿Por qué LSTM resuelve el problema de vanishing gradient?**

*Respuesta:* LSTM usa un **cell state** que fluye directamente a través de los timesteps con suma (no multiplicación). Las compuertas controlan qué agregar/remover del cell state, pero los gradientes fluyen sin multiplicarse repetidamente por matrices de pesos, evitando que se desvanezcan.

**2. ¿Qué hace cada compuerta (gate) en una LSTM?**

*Respuesta:*
- **Forget gate**: Decide qué información del cell state anterior descartar (qué olvidar)
- **Input gate**: Decide qué nueva información agregar al cell state
- **Output gate**: Decide qué parte del cell state exponer como hidden state

Cada una aprende cuándo activarse basándose en el input actual y el hidden state anterior.

**3. ¿Por qué necesitamos padding? ¿Qué problema resuelve?**

*Respuesta:* Las redes neuronales requieren inputs de tamaño fijo, pero las frases tienen longitudes variables. Padding rellena secuencias cortas con ceros para que todas tengan la misma longitud. Esto permite procesarlas en batches eficientemente.

**4. ¿Cuál es la diferencia fundamental entre Bag of Words y procesamiento secuencial?**

*Respuesta:*
- **BoW**: Ignora orden, solo cuenta ocurrencias. "No me gusta" = "Me gusta no"
- **Secuencial (LSTM)**: Procesa palabra por palabra en orden. Captura que "no" antes de "me gusta" invierte el sentimiento

**5. ¿Qué aprende la capa Embedding?**

*Respuesta:* Aprende representaciones vectoriales densas de palabras donde palabras con significados similares tienen vectores cercanos en el espacio de embeddings. Durante el entrenamiento, ajusta estos vectores para que sean útiles para la tarea específica (en nuestro caso, clasificación de sentimientos).

### Preguntas Técnicas

**6. En el código, ¿por qué vocab_size = len(tokenizer.word_index) + 1?**

*Respuesta:* El +1 es porque Keras reserva el índice 0 para padding. El tokenizer asigna índices desde 1 en adelante a las palabras. Entonces si tenemos 100 palabras únicas, word_index va de 1 a 100, y necesitamos 101 posiciones (0-100) en la capa Embedding.

**7. ¿Qué pasa si una palabra nueva (no en el vocabulario) aparece en producción?**

*Respuesta:* El tokenizer la reemplaza por el token OOV (Out Of Vocabulary) que configuramos con `oov_token="<OOV>"`. Este token tiene su propio embedding aprendido que representa "palabra desconocida". Es importante incluir el oov_token durante entrenamiento para que el modelo aprenda a manejarlo.

**8. ¿Por qué LSTM devuelve solo el hidden state final y no todos?**

*Respuesta:* Porque `return_sequences=False` (default). Para clasificación, solo necesitamos la representación final que "resume" toda la secuencia. Si configuramos `return_sequences=True`, devolvería el hidden state en cada timestep, útil para apilar LSTMs o tareas como NER donde necesitamos clasificar cada token.

**9. En la capa Embedding, ¿qué significan los parámetros input_dim, output_dim, input_length?**

*Respuesta:*
- `input_dim`: Tamaño del vocabulario (cuántas palabras diferentes puede manejar)
- `output_dim`: Dimensión del vector de embedding (ej: 16, 100, 300)
- `input_length`: Longitud de las secuencias de entrada (después del padding)

**10. ¿Qué hace padding='post' vs. padding='pre'?**

*Respuesta:*
- `padding='post'`: Agrega ceros al **final**: [5, 2, 8, 0, 0, 0]
- `padding='pre'`: Agrega ceros al **inicio**: [0, 0, 0, 5, 2, 8]

Para LSTM, `post` suele ser mejor porque las palabras relevantes quedan al principio de la secuencia (donde la LSTM tiene mejor memoria).

### Preguntas de Aplicación

**11. Si tuvieras un dataset de 50,000 reseñas, ¿qué cambios harías al código?**

*Respuesta:*
1. **Train/val/test split**: 70/15/15 para evaluación honesta
2. **Batch size mayor**: 32 o 64 (más eficiente)
3. **Validation en fit()**: Monitorear overfitting
4. **Early stopping**: Callback para detener si val_loss no mejora
5. **Modelo más grande**: embedding_dim=100, lstm_units=128
6. **Dropout**: Para regularización
7. **Learning rate scheduling**: Reducir lr gradualmente

**12. ¿Cómo implementarías un modelo Seq2Seq para traducción español→inglés?**

*Respuesta:*
```python
# Encoder: Procesa español
encoder_inputs = Input(shape=(None,))
encoder_embedding = Embedding(vocab_size_es, 256)(encoder_inputs)
encoder_lstm = LSTM(512, return_state=True)
encoder_outputs, state_h, state_c = encoder_lstm(encoder_embedding)
encoder_states = [state_h, state_c]

# Decoder: Genera inglés
decoder_inputs = Input(shape=(None,))
decoder_embedding = Embedding(vocab_size_en, 256)(decoder_inputs)
decoder_lstm = LSTM(512, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(
    decoder_embedding,
    initial_state=encoder_states  # ← Contexto del encoder
)
decoder_dense = Dense(vocab_size_en, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
```

**13. ¿Cómo cargarías embeddings pre-entrenados de Word2Vec en este modelo?**

*Respuesta:*
```python
from gensim.models import KeyedVectors

# 1. Cargar Word2Vec
w2v = KeyedVectors.load_word2vec_format('GoogleNews-vectors.bin', binary=True)

# 2. Crear matriz de embeddings
embedding_matrix = np.zeros((vocab_size, embedding_dim))
for word, idx in tokenizer.word_index.items():
    if word in w2v:
        embedding_matrix[idx] = w2v[word]

# 3. Configurar capa Embedding
Embedding(
    vocab_size,
    embedding_dim,
    weights=[embedding_matrix],
    trainable=False  # Congelar o permitir fine-tuning
)
```

**14. Si el modelo predice siempre la misma clase, ¿qué revisarías?**

*Respuesta:*
1. **Dataset desbalanceado**: Verificar distribución de clases
2. **Learning rate muy alto**: Probablemente divergió
3. **Inicialización de pesos**: Revisar si hay NaNs
4. **Preprocesamiento incorrecto**: ¿El tokenizer se aplicó bien?
5. **Loss function incorrecta**: Verificar que es binary_crossentropy
6. **Modelo muy simple**: Aumentar capacidad (más unidades)

**15. Diseñá una arquitectura LSTM profunda con 3 capas LSTM apiladas.**

*Respuesta:*
```python
modelo = Sequential([
    Embedding(vocab_size, 64, input_length=maxlen),
    
    # LSTM 1: return_sequences=True para alimentar siguiente LSTM
    LSTM(128, return_sequences=True, dropout=0.2),
    
    # LSTM 2: También return_sequences=True
    LSTM(64, return_sequences=True, dropout=0.2),
    
    # LSTM 3: return_sequences=False (solo hidden state final)
    LSTM(32, dropout=0.2),
    
    # Clasificación
    Dense(1, activation='sigmoid')
])
```
**Trade-off:** Más parámetros = más capacidad pero más riesgo de overfitting.

---

## Ejercicios propuestos

### Ejercicio 1: `Bidirectional(LSTM)`
Probá una versión bidireccional y compará tiempo de entrenamiento, validación y pequeña prueba externa.

### Ejercicio 2: GRU vs. LSTM
Reemplazá `LSTM` por `GRU` y registrá qué cambia en velocidad y rendimiento.

### Ejercicio 3: embeddings preentrenados
Intentá reemplazar la capa de embeddings aprendidos por embeddings preentrenados y compará si la convergencia mejora.

### Ejercicio 4: análisis de errores
Identificá qué frases generan más confusión. Prestá especial atención a negaciones, señales mixtas y vocabulario que no estuvo en el corpus.

## Cierre

Este notebook mostró el paso desde una red feedforward a una arquitectura secuencial. La gran diferencia no está solo en la cantidad de parámetros, sino en la forma de leer el dato: ahora el orden de las palabras sí importa.

En el próximo cuaderno vamos a dar otro salto importante: usaremos transformers preentrenados con HuggingFace.
